# Portfolio VaR + Stress Testing

Notebook-first workflow for the 3-stock risk engine. Each section calls the same modular services as `main.py`, then renders tables and charts inline.

In [15]:
# Imports: load libraries and project services used throughout notebook.

import numpy as np
import pandas as pd
import plotly.express as px
import plotly.graph_objects as go

from services.backtesting import RollingVaRBreachBacktestService
from services.data_fetch import PriceDataFetchService, PriceDataRequest
from services.returns import ReturnCalculatorService
from services.scenarios import (
    CorrelationSpikeScenario,
    EquityShock2008Scenario,
    ScenarioInput,
    TechDrawdownScenario,
)
from services.var import (
    HistoricalVaRService,
    MonteCarloVaRService,
    ParametricVaRService,
)

pd.options.display.float_format = "{:.6f}".format

## Configuration

In [16]:
# Config: define tickers, weights, portfolio value, and start date.

TICKERS = ["AAPL", "GOOGL", "MSFT"]
WEIGHTS = np.array([0.40, 0.35, 0.25])
PORTFOLIO_VALUE = 1_000_000
START_DATE = "2020-01-01"

portfolio_config = pd.DataFrame(
    {
        "ticker": TICKERS,
        "weight": WEIGHTS,
        "allocation_dollar": WEIGHTS * PORTFOLIO_VALUE,
    }
)
portfolio_config

,ticker,weight,allocation_dollar
0,AAPL,0.400000,400000.000000
1,GOOGL,0.350000,350000.000000
2,MSFT,0.250000,250000.000000


## Fetch Prices

In [17]:
# Data fetch: pull adjusted close prices through PriceDataFetchService.

data_service = PriceDataFetchService()
request = PriceDataRequest(tickers=TICKERS, start_date=START_DATE)

prices = data_service.fetch_prices(request)
data_service.save_prices(prices, "prices.csv")

prices.tail()

Ticker,AAPL,GOOGL,MSFT
Date,,,
2026-05-01,280.140015,385.690002,414.440002
2026-05-04,276.829987,383.250000,413.619995
2026-05-05,284.179993,388.429993,411.380005
2026-05-06,287.510010,398.040009,413.959991
2026-05-07,287.440002,397.989990,420.769989


In [18]:
# Price chart: visualize adjusted close price history.

px.line(prices, title="Adjusted Close Prices", labels={"value": "Price", "Date": "Date"})

## Daily Returns

In [19]:
# Return calc: compute stock returns and weighted portfolio returns through ReturnCalculatorService.

return_service = ReturnCalculatorService()

returns = return_service.calculate_returns(prices)
return_service.save_returns(returns, "returns.csv")

portfolio_returns = return_service.calculate_portfolio_returns(returns, WEIGHTS)
return_service.save_portfolio_returns(portfolio_returns, "portfolio_returns.csv")

returns.tail()

Ticker,AAPL,GOOGL,MSFT
Date,,,
2026-05-01,0.032394,0.002313,0.016332
2026-05-04,-0.011816,-0.006326,-0.001979
2026-05-05,0.026551,0.013516,-0.005416
2026-05-06,0.011718,0.024741,0.006272
2026-05-07,-0.000243,-0.000126,0.016451


In [20]:
# Portfolio returns chart: visualize daily portfolio return path.

px.line(portfolio_returns, title="Portfolio Daily Returns", labels={"value": "Return", "Date": "Date"})

In [21]:
# Distribution chart: inspect empirical portfolio return distribution.

fig = go.Figure()
fig.add_trace(go.Histogram(x=portfolio_returns, nbinsx=80, name="Portfolio returns"))
fig.update_layout(title="Portfolio Return Distribution", xaxis_title="Daily return", yaxis_title="Count")
fig

## VaR Calculations

In [22]:
# VaR calc: compute Historical, Parametric, and Monte Carlo VaR through services.

historical_var_service = HistoricalVaRService()
parametric_var_service = ParametricVaRService()
monte_carlo_var_service = MonteCarloVaRService()

historical_var_95 = historical_var_service.calculate_var(portfolio_returns, PORTFOLIO_VALUE, 0.95)
historical_var_99 = historical_var_service.calculate_var(portfolio_returns, PORTFOLIO_VALUE, 0.99)
parametric_var_95 = parametric_var_service.calculate_var(returns, WEIGHTS, PORTFOLIO_VALUE, 0.95)
parametric_var_99 = parametric_var_service.calculate_var(returns, WEIGHTS, PORTFOLIO_VALUE, 0.99)
monte_carlo_var = monte_carlo_var_service.calculate_var(
    returns,
    WEIGHTS,
    portfolio_value=PORTFOLIO_VALUE,
    num_simulations=10_000,
    seed=42,
)

var_summary = pd.DataFrame(
    [
        ["Historical", "95%", historical_var_95.var_return, historical_var_95.var_dollar, None],
        ["Historical", "99%", historical_var_99.var_return, historical_var_99.var_dollar, None],
        ["Parametric", "95%", None, parametric_var_95.var_dollar, parametric_var_95.portfolio_volatility],
        ["Parametric", "99%", None, parametric_var_99.var_dollar, parametric_var_99.portfolio_volatility],
        ["Monte Carlo", "95%", monte_carlo_var.var_95_return, monte_carlo_var.var_95_dollar, None],
        ["Monte Carlo", "99%", monte_carlo_var.var_99_return, monte_carlo_var.var_99_dollar, None],
    ],
    columns=["method", "confidence", "var_return", "var_dollar", "portfolio_volatility"],
)
var_summary

,method,confidence,var_return,var_dollar,portfolio_volatility
0,Historical,95%,-0.026083,26083.320513,NaN
1,Historical,99%,-0.044984,44984.216238,NaN
2,Parametric,95%,NaN,28476.063571,0.017312
3,Parametric,99%,NaN,40274.240130,0.017312
4,Monte Carlo,95%,-0.027705,27705.411987,NaN
5,Monte Carlo,99%,-0.038711,38711.024247,NaN


In [23]:
# VaR chart: compare dollar VaR across methods and confidence levels.

px.bar(
    var_summary,
    x="method",
    y="var_dollar",
    color="confidence",
    barmode="group",
    title="VaR Dollar Comparison",
)

In [24]:
# VaR cutoff chart: show historical 95% and 99% cutoff returns.

fig = go.Figure()
fig.add_trace(go.Histogram(x=portfolio_returns, nbinsx=80, name="Portfolio returns"))
fig.add_vline(x=historical_var_95.var_return, line_color="orange", annotation_text="Historical 95%")
fig.add_vline(x=historical_var_99.var_return, line_color="red", annotation_text="Historical 99%")
fig.update_layout(title="Historical VaR Cutoffs", xaxis_title="Daily return", yaxis_title="Count")
fig

## Stress Scenarios

In [25]:
# Scenario calc: run all Scenario implementations through common ScenarioInput.

scenarios = [
    EquityShock2008Scenario(),
    TechDrawdownScenario(),
    CorrelationSpikeScenario(parametric_var_service),
]
scenario_input = ScenarioInput(
    returns=returns,
    weights=WEIGHTS,
    portfolio_value=PORTFOLIO_VALUE,
)
scenario_results = [scenario.run(scenario_input) for scenario in scenarios]

scenario_rows = []
for result in scenario_results:
    row = {"scenario": result.name}
    row.update(result.metrics)
    scenario_rows.append(row)

scenario_summary = pd.DataFrame(scenario_rows)
scenario_summary

,scenario,portfolio_return,portfolio_pnl_dollar,portfolio_loss_dollar,stressed_corr,portfolio_vol,var_95_dollar,var_99_dollar
0,2008-style Equity Shock,-0.071500,-71500.000000,71500.000000,NaN,NaN,NaN,NaN
1,Tech Drawdown,-0.100000,-100000.000000,100000.000000,NaN,NaN,NaN,NaN
2,Correlation Spike Scenario,NaN,NaN,NaN,0.850000,0.018763,30862.619540,43649.591780


In [26]:
# Scenario chart: compare dollar impact across stress scenarios.

scenario_chart = scenario_summary.melt(
    id_vars="scenario",
    value_vars=[col for col in ["portfolio_loss_dollar", "var_95_dollar", "var_99_dollar"] if col in scenario_summary.columns],
    var_name="metric",
    value_name="dollar_value",
).dropna()

px.bar(
    scenario_chart,
    x="scenario",
    y="dollar_value",
    color="metric",
    barmode="group",
    title="Stress Scenario Dollar Impact",
)

## Rolling VaR Breach Backtest

Use previous 250 trading days to estimate rolling Historical VaR(95), then count days where realized portfolio return breaches that cutoff. Expected breach rate is about 5%.

In [27]:
# Backtest calc: compute rolling 95% historical VaR exceedances using prior 250 days.

backtest_service = RollingVaRBreachBacktestService()
backtest_result = backtest_service.run(
    portfolio_returns,
    confidence_level=0.95,
    window=250,
)

backtest_summary = pd.DataFrame(
    [
        {
            "confidence_level": backtest_result.confidence_level,
            "window": backtest_result.window,
            "observations": backtest_result.observations,
            "number_of_breaches": backtest_result.number_of_breaches,
            "exceedance_rate": backtest_result.exceedance_rate,
            "expected_exceedance_rate": backtest_result.expected_breach_rate,
            "expected_breaches": backtest_result.expected_breaches,
            "comment": backtest_result.comment,
        }
    ]
)
backtest_summary


,confidence_level,window,observations,number_of_breaches,exceedance_rate,expected_exceedance_rate,expected_breaches,comment
0,0.950000,250,1344,70,0.052083,0.050000,67.200000,Exceedance rate is close to expected rate.


In [28]:
# Backtest chart: plot realized returns, rolling VaR cutoff, and exceedance points.

backtest_data = backtest_result.backtest_data.copy()
exceedance_mask = backtest_data["breach"].astype(bool)
exceedances = backtest_data.loc[exceedance_mask]

fig = go.Figure()
fig.add_trace(
    go.Scatter(
        x=backtest_data.index,
        y=backtest_data["portfolio_return"],
        mode="lines",
        name="Portfolio return",
        line={"color": "#2563eb", "width": 1},
        hovertemplate="Date=%{x}<br>Return=%{y:.2%}<extra></extra>",
    )
)
fig.add_trace(
    go.Scatter(
        x=backtest_data.index,
        y=backtest_data["rolling_var"],
        mode="lines",
        name="Rolling VaR 95%",
        line={"color": "#f97316", "width": 2, "dash": "dash"},
        hovertemplate="Date=%{x}<br>VaR cutoff=%{y:.2%}<extra></extra>",
    )
)
fig.add_trace(
    go.Scatter(
        x=exceedances.index,
        y=exceedances["portfolio_return"],
        mode="markers",
        marker={"color": "#dc2626", "size": 7},
        name="Exceedances",
        hovertemplate="Date=%{x}<br>Return=%{y:.2%}<extra></extra>",
    )
)
fig.add_hline(y=0, line_color="#94a3b8", line_width=1)
fig.update_layout(
    title=(
        "Rolling Historical VaR(95) Backtest "
        f"- {backtest_result.number_of_breaches} exceedances, "
        f"{backtest_result.exceedance_rate:.2%} rate"
    ),
    xaxis_title="Date",
    yaxis_title="Daily return",
    yaxis_tickformat=".1%",
    legend_title_text="Series",
)
fig


## Key Assumptions

- Historical VaR uses empirical portfolio return quantiles.
- Parametric VaR assumes daily returns are normally distributed.
- Monte Carlo uses multivariate normal draws from historical mean and covariance.
- Correlation spike keeps individual volatility fixed and forces pairwise correlation to 0.85.
- Shock scenarios apply one-day portfolio return shocks directly.

- Rolling VaR backtest uses prior-window data only to avoid lookahead bias.
